# 03 - Feature Engineering & Business Intelligence
**Parts G + I:** date/time/maths features, encoding, scaling, RFM segmentation, KPIs and sales analysis.
Outputs `datasets/processed/sales_features.csv` and `customer_rfm.csv`, the inputs for Task 3 (machine learning).

In [1]:
import sys
sys.path.insert(0, "../scripts")          # reuse the project modules
import pandas as pd, numpy as np
from sqlalchemy import text
import db
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 30
engine = db.get_engine()
import plotly.express as px
import features as F, viz
df = F.load_sales()
print(df.shape)

(5905, 18)


## 1. Feature engineering

In [2]:
feats = F.engineer_sales_features(df)
print("Rows:", len(feats), "| Columns:", feats.shape[1], "(from", df.shape[1], ")")
new_cols = [c for c in feats.columns if c not in df.columns]
print(len(new_cols), "new features:\n", new_cols)

Rows: 5905 | Columns: 58 (from 18 )
40 new features:
 ['week_of_year', 'day_of_month', 'is_weekend', 'is_month_end', 'is_holiday_season', 'month_sin', 'month_cos', 'days_since_start', 'gross_amount', 'discount_amount', 'log_revenue', 'price_tier', 'is_bulk_order', 'customer_purchase_seq', 'days_since_prev_purchase', 'customer_revenue_before', 'prev3_avg_revenue', 'product_code', 'cat_Clothing', 'cat_Furniture', 'cat_Home & Kitchen', 'cat_Office Supplies', 'cat_Technology', 'reg_Central', 'reg_East', 'reg_North', 'reg_South', 'reg_West', 'pay_Cash', 'pay_Credit Card', 'pay_Debit Card', 'pay_Paypal', 'quantity_minmax', 'quantity_z', 'unit_price_minmax', 'unit_price_z', 'discount_minmax', 'discount_z', 'revenue_minmax', 'revenue_z']


In [3]:
feats[["order_date", "is_weekend", "is_holiday_season", "month_sin", "month_cos", "gross_amount",
       "discount_amount", "price_tier", "customer_purchase_seq", "days_since_prev_purchase",
       "customer_revenue_before", "prev3_avg_revenue"]].head(8)

,order_date,is_weekend,is_holiday_season,month_sin,month_cos,gross_amount,discount_amount,price_tier,customer_purchase_seq,days_since_prev_purchase,customer_revenue_before,prev3_avg_revenue
0,2024-01-01,0,0,0.50,0.87,51.72,0.00,Standard,1,0,0.00,0.00
1,2024-01-01,0,0,0.50,0.87,530.24,26.51,Luxury,1,0,0.00,0.00
2,2024-01-01,0,0,0.50,0.87,192.42,0.00,Premium,1,0,0.00,0.00
3,2024-01-01,0,0,0.50,0.87,81.91,0.00,Standard,1,0,0.00,0.00
4,2024-01-01,0,0,0.50,0.87,47.44,7.12,Budget,1,0,0.00,0.00
5,2024-01-01,0,0,0.50,0.87,638.52,31.93,Luxury,1,0,0.00,0.00
6,2024-01-02,0,0,0.50,0.87,153.90,7.69,Standard,1,0,0.00,0.00
7,2024-01-02,0,0,0.50,0.87,101.28,10.13,Standard,1,0,0.00,0.00


In [4]:
# Encoding and scaling
feats.filter(regex="^(cat_|reg_|pay_)").head(3)

,cat_Clothing,cat_Furniture,cat_Home & Kitchen,cat_Office Supplies,cat_Technology,reg_Central,reg_East,reg_North,reg_South,reg_West,pay_Cash,pay_Credit Card,pay_Debit Card,pay_Paypal
0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
1,0,0,0,0,1,0,0,0,0,1,0,1,0,0
2,1,0,0,0,0,1,0,0,0,0,0,1,0,0


In [5]:
feats[["revenue", "revenue_minmax", "revenue_z", "log_revenue"]].describe().round(3)

,revenue,revenue_minmax,revenue_z,log_revenue
count,"5,905.00","5,905.00","5,905.00","5,905.00"
mean,257.45,0.07,0.00,4.98
std,330.52,0.09,1.00,1.07
min,6.94,0.00,-0.76,2.07
25%,67.14,0.02,-0.58,4.22
50%,135.32,0.04,-0.37,4.92
75%,313.57,0.09,0.17,5.75
max,"3,610.14",1.00,10.14,8.19


**Why scaling and log?** Revenue is right-skewed (mean far above median). `log1p` compresses it and min-max/z-score put features on comparable scales, which many ML models need.

**Leakage check:** customer-history features (`customer_revenue_before`, `prev3_avg_revenue`) only use *earlier* rows, so a model can never "see the future".

In [6]:
print("Skewness before log:", round(feats.revenue.skew(), 2), "| after log:", round(feats.log_revenue.skew(), 2))
print("Missing values in feature table:", feats.isna().sum().sum())

Skewness before log: 3.02 | after log: 0.18
Missing values in feature table: 0


## 2. Customer segmentation with RFM

In [7]:
rfm = F.compute_rfm(df)
rfm.head(8)

,customer_id,customer_name,region,last_order,frequency,monetary,recency_days,r_score,f_score,m_score,rfm_score,segment
0,CUST-1269,Liam Sharma,East,2025-12-31,262,"65,795.85",1,5,5,5,555,Champions
1,CUST-1054,Emma Wang,East,2025-12-31,162,"48,958.85",1,5,5,5,555,Champions
2,CUST-1099,Arjun Silva,East,2025-12-30,123,"41,390.61",2,5,5,5,555,Champions
3,CUST-1065,Olivia Garcia,East,2025-12-30,104,"29,109.70",2,5,5,5,555,Champions
4,CUST-1008,Liam Patel,South,2025-12-21,85,"26,093.40",11,4,5,5,455,Champions
5,CUST-1073,Aarav Reddy,South,2025-12-26,80,"25,531.96",6,4,5,5,455,Champions
6,CUST-1176,Sneha Smith,East,2025-12-28,93,"23,739.02",4,5,5,5,555,Champions
7,CUST-1169,Vikram Silva,East,2025-12-29,78,"22,552.15",3,5,5,5,555,Champions


In [8]:
seg = (rfm.groupby("segment").agg(customers=("customer_id", "count"), revenue=("monetary", "sum"),
       avg_recency_days=("recency_days", "mean"), avg_orders=("frequency", "mean")).round(1))
seg["revenue_share_%"] = (seg.revenue / seg.revenue.sum() * 100).round(1)
seg.loc[F.SEGMENT_ORDER]

,customers,revenue,avg_recency_days,avg_orders,revenue_share_%
segment,,,,,
Champions,79,"976,690.40",6.50,45.80,64.20
Loyal,58,"279,743.60",21.50,18.80,18.40
Needs Attention,63,"136,022.60",42.90,8.90,8.90
At Risk (high value),8,"49,288.60",71.20,22.20,3.20
Hibernating,84,"78,518.80",169.30,4.50,5.20


In [9]:
viz.rfm_segments(rfm)

In [10]:
viz.rfm_scatter(rfm)

## 3. KPIs and business analysis

In [11]:
total = df.revenue.sum(); orders = df.order_id.nunique()
kpi = pd.Series({"Total revenue": total, "Orders": orders, "Customers": df.customer_id.nunique(),
                 "Avg order value": total / orders, "Units sold": df.quantity.sum()})
kpi.map(lambda v: f"{v:,.0f}")

Total revenue      1,520,264
Orders                 5,825
Customers                292
Avg order value          261
Units sold            13,116
dtype: str

In [12]:
viz.monthly_trend(df)

In [13]:
viz.category_treemap(df)

In [14]:
viz.region_category_heatmap(df)

In [15]:
viz.pareto(df)

## 4. Save engineered datasets (CSV + back into PostgreSQL)

In [16]:
feats, rfm = F.build_and_save(engine)
db.read_sql("SELECT COUNT(*) AS rows_in_ml_sales_features FROM ml_sales_features")

Feature table: 5,905 rows x 58 columns
RFM table: 292 customers | segments: {'Hibernating': 84, 'Champions': 79, 'Needs Attention': 63, 'Loyal': 58, 'At Risk (high value)': 8}


,rows_in_ml_sales_features
0,5905


## 5. Business insights (computed)

In [17]:
yrs = df.groupby("year").revenue.sum()
cust = df.groupby("customer_id").revenue.sum().sort_values(ascending=False)
champ = seg.loc["Champions"]; risk = seg.loc["At Risk (high value)"]
print(f"1. Revenue ${total:,.0f}; {((yrs.iloc[-1] / yrs.iloc[0]) - 1) * 100:+.1f}% growth {yrs.index[0]} to {yrs.index[-1]}.")
print(f"2. Champions: {int(champ.customers)} customers = {champ['revenue_share_%']}% of revenue.")
print(f"3. At-risk high-value customers: {int(risk.customers)} (${risk.revenue:,.0f} historic revenue) -> win-back target.")
print(f"4. Top 10% of customers = {cust.head(round(len(cust) * .1)).sum() / cust.sum() * 100:.1f}% of revenue.")
print(f"5. Q4 share of revenue: {df[df.quarter == 4].revenue.sum() / total * 100:.1f}% -> plan stock and campaigns around it.")

1. Revenue $1,520,264; +4.6% growth 2024 to 2025.
2. Champions: 79 customers = 64.2% of revenue.
3. At-risk high-value customers: 8 ($49,289 historic revenue) -> win-back target.
4. Top 10% of customers = 41.5% of revenue.
5. Q4 share of revenue: 33.5% -> plan stock and campaigns around it.
